# ---------- Data loading notebook ----------

The goal of this notebook is to load and combine London bike-sharing trip data from 2024 and 2025.

Author: Artur Werys

In [1]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as et

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [2]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"

DATA_2024_DIR = DATA_DIR / "2024"
DATA_2025_DIR = DATA_DIR / "2025"

STATIONS_FILE = DATA_DIR / "stations.xml"

## ---------- Loading CSV data function ----------


In [3]:
def load_data(folder_path):
    files = list(folder_path.glob("*.csv"))
    dataframes = []

    for file in files:
        df = pd.read_csv(file)
        dataframes.append(df)

    combined_data = pd.concat(dataframes, ignore_index=True)

    return combined_data

## ---------- Loading data from choosen years ----------


In [4]:
print("--- Loading data from 2025... ---")

data_2025 = load_data(DATA_2025_DIR)

print(data_2025.head())
print("Number of trips in 2025:", len(data_2025))


print("--- Loading data from 2024... ---")

data_2024 = load_data(DATA_2024_DIR)

print(data_2024.head())
print("Number of trips in 2024:", len(data_2024))

--- Loading data from 2025... ---
      Number        Start date  Start station number  \
0  145666816  2025-01-14 23:59                  1043   
1  145666817  2025-01-14 23:59                300015   
2  145666818  2025-01-14 23:59                  1068   
3  145666819  2025-01-14 23:59                  1159   
4  145666812  2025-01-14 23:58                200048   

                      Start station          End date  End station number  \
0        Museum of London, Barbican  2025-01-15 00:13            200149.0   
1          Binfield Road, Stockwell  2025-01-15 00:04            300229.0   
2  Norton Folgate, Liverpool Street  2025-01-15 00:10              1051.0   
3         Berry Street, Clerkenwell  2025-01-15 00:10              1007.0   
4          Page Street, Westminster  2025-01-15 00:14              1058.0   

                        End station  Bike number  Bike model Total duration  \
0           Watney Street, Shadwell        55223     CLASSIC        14m 30s   
1       

### ---------- Combining datasets ----------


In [5]:
combined_trip_data = pd.concat(
    [data_2024, data_2025],
    ignore_index=True
)

### ---------- Renaming columns ----------


In [6]:
combined_trip_data = combined_trip_data.rename(columns={
    "Number": "trip_id",
    "Start date": "start_date",
    "End date": "end_date",
    "Start station number": "start_station_id",
    "Start station": "start_station_name",
    "End station number": "end_station_id",
    "End station": "end_station_name",
    "Bike number": "bike_id",
    "Bike model": "bike_model",
    "Total duration": "total_duration",
    "Total duration (ms)": "total_duration_ms",
})

combined_trip_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms
0,145207079,2024-12-14 23:59,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,29203.0
1,145207080,2024-12-14 23:59,1133,"Baylis Road, Waterloo",2024-12-15 00:26,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,1620164.0
2,145207081,2024-12-14 23:59,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,1096981.0
3,145207082,2024-12-14 23:59,1112,"Nutford Place, Marylebone",2024-12-15 00:12,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,783763.0
4,145207083,2024-12-14 23:59,1122,"Ashley Place, Victoria",2024-12-15 00:04,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,322155.0


### ---------- Checking for missing values in combined data ----------


In [7]:
print("Missing values in each column:")
print(combined_trip_data.isna().sum())

len_before_dropping = len(combined_trip_data)

print("Dropping rows with missing data...")

combined_trip_data = combined_trip_data.dropna(
    subset=["end_date", "end_station_id", "bike_id"]
)

len_after_dropping = len(combined_trip_data)

dropped_rows_count = len_before_dropping - len_after_dropping

print("Missing values in each column after dropping rows with missing data:")
print(combined_trip_data.isna().sum())

print(f"Number of dropped rows: {dropped_rows_count}")

Missing values in each column:
trip_id                 0
start_date              0
start_station_id        0
start_station_name      0
end_date              184
end_station_id        184
end_station_name      184
bike_id                 1
bike_model              0
total_duration        184
total_duration_ms     184
dtype: int64
Dropping rows with missing data...
Missing values in each column after dropping rows with missing data:
trip_id               0
start_date            0
start_station_id      0
start_station_name    0
end_date              0
end_station_id        0
end_station_name      0
bike_id               0
bike_model            0
total_duration        0
total_duration_ms     0
dtype: int64
Number of dropped rows: 185


## ---------- How much data was removed? ----------

In [14]:
dropped_percent = dropped_rows_count / len_before_dropping * 100
remaining_percent = len_after_dropping / len_before_dropping * 100

print(f"Number of rows before dropping: {len_before_dropping:.2e}")
print(f"Number of rows after dropping: {len_after_dropping:.2e}")
print(f"Number of dropped rows: {dropped_rows_count:.2e}")

print(f"Percentage of dropped rows: {dropped_percent:.3f}%")
print(f"Percentage of remaining rows: {remaining_percent:.3f}%")

Number of rows before dropping: 1.78e+07
Number of rows after dropping: 1.78e+07
Number of dropped rows: 1.85e+02
Percentage of dropped rows: 0.001%
Percentage of remaining rows: 99.999%


## ---------- Loading station data from XML ----------


In [15]:
tree = et.parse(STATIONS_FILE)
root = tree.getroot()

stations_data = []

for station in root.findall("station"):

    station_data = [
        station.find("name").text.strip(),
        station.find("lat").text,
        station.find("long").text
    ]

    stations_data.append(station_data)

stations_df = pd.DataFrame(
    stations_data,
    columns=[
        "station_name",
        "latitude",
        "longitude"
    ]
)

print("--- Station data loaded from XML ---")
print(stations_df.head())
print("Number of stations:", len(stations_df))

--- Station data loaded from XML ---
                           station_name     latitude     longitude
0            River Street , Clerkenwell  51.52916347  -0.109970527
1        Phillimore Gardens, Kensington  51.49960695  -0.197574246
2  Christopher Street, Liverpool Street  51.52128377  -0.084605692
3       St. Chad's Street, King's Cross  51.53005939  -0.120973687
4         Sedding Street, Sloane Square     51.49313     -0.156876
Number of stations: 801


### ---------- Merging stations from trip data with geograpical data ----------


In [16]:
combined_trip_data = pd.merge(combined_trip_data, stations_df, left_on="start_station_name", right_on="station_name", how ="left")
combined_trip_data = combined_trip_data.rename(columns={
    "latitude": "start_lat",
    "longitude": "start_lon"
})

combined_trip_data = combined_trip_data.drop(columns=["station_name"])
combined_trip_data.head()

combined_trip_data = pd.merge(combined_trip_data, stations_df, left_on="end_station_name", right_on="station_name", how ="left")
combined_trip_data = combined_trip_data.rename(columns={
    "latitude": "end_lat",
    "longitude": "end_lon"
})

combined_trip_data = combined_trip_data.drop(columns=["station_name"])
combined_trip_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon
0,145207079,2024-12-14 23:59,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,29203.0,51.50630441,-0.087262995,51.50630441,-0.087262995
1,145207080,2024-12-14 23:59,1133,"Baylis Road, Waterloo",2024-12-15 00:26,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,1620164.0,51.50144456,-0.110699309,51.519265,-0.021345
2,145207081,2024-12-14 23:59,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,1096981.0,51.49792478,-0.183834706,51.50035306,-0.217515071
3,145207082,2024-12-14 23:59,1112,"Nutford Place, Marylebone",2024-12-15 00:12,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,783763.0,51.5165179,-0.164393768,51.53430039,-0.1680743
4,145207083,2024-12-14 23:59,1122,"Ashley Place, Victoria",2024-12-15 00:04,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,322155.0,51.49616092,-0.140947636,51.493978,-0.127554


### ---------- Deleting trips without geographical data ----------


In [17]:
final_trip_data = combined_trip_data.dropna(subset=["start_lat", "start_lon", "end_lat", "end_lon"])

# How many trips were deleted?
print(f"Percentage of trips deleted: {((len(combined_trip_data) - len(final_trip_data)) / len(combined_trip_data) * 100):.2f}%")


Percentage of trips deleted: 2.23%


## ---------- Saving final trip data as parquet ----------


In [ ]:
final_trip_data["start_date"] = pd.to_datetime(
    final_trip_data["start_date"],
    format="mixed",
    dayfirst=True
)

final_trip_data["end_date"] = pd.to_datetime(
    final_trip_data["end_date"],
    format="mixed",
    dayfirst=True
)

final_trip_data.to_parquet(DATA_DIR / "final_trip_data.parquet")

/var/folders/8p/j63xl3c17439n25jbp9cyjg40000gn/T/ipykernel_3127/2951280931.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_trip_data["start_date"] = pd.to_datetime(
/var/folders/8p/j63xl3c17439n25jbp9cyjg40000gn/T/ipykernel_3127/2951280931.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_trip_data["end_date"] = pd.to_datetime(


In [ ]:
final_trip_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon
0,145207079,2024-12-14 23:59:00,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59:00,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,29203.0,51.50630441,-0.087262995,51.50630441,-0.087262995
1,145207080,2024-12-14 23:59:00,1133,"Baylis Road, Waterloo",2024-12-15 00:26:00,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,1620164.0,51.50144456,-0.110699309,51.519265,-0.021345
2,145207081,2024-12-14 23:59:00,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17:00,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,1096981.0,51.49792478,-0.183834706,51.50035306,-0.217515071
3,145207082,2024-12-14 23:59:00,1112,"Nutford Place, Marylebone",2024-12-15 00:12:00,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,783763.0,51.5165179,-0.164393768,51.53430039,-0.1680743
4,145207083,2024-12-14 23:59:00,1122,"Ashley Place, Victoria",2024-12-15 00:04:00,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,322155.0,51.49616092,-0.140947636,51.493978,-0.127554


In [ ]:
final_trip_data.head(-5)

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon
0,145207079,2024-12-14 23:59:00,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59:00,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,29203.0,51.50630441,-0.087262995,51.50630441,-0.087262995
1,145207080,2024-12-14 23:59:00,1133,"Baylis Road, Waterloo",2024-12-15 00:26:00,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,1620164.0,51.50144456,-0.110699309,51.519265,-0.021345
2,145207081,2024-12-14 23:59:00,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17:00,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,1096981.0,51.49792478,-0.183834706,51.50035306,-0.217515071
3,145207082,2024-12-14 23:59:00,1112,"Nutford Place, Marylebone",2024-12-15 00:12:00,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,783763.0,51.5165179,-0.164393768,51.53430039,-0.1680743
4,145207083,2024-12-14 23:59:00,1122,"Ashley Place, Victoria",2024-12-15 00:04:00,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,322155.0,51.49616092,-0.140947636,51.493978,-0.127554
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17823198,146319209,2025-02-15 00:01:00,1163,"Wardour Street, Soho",2025-02-15 00:23:00,1210.0,"Nevern Place, Earl's Court",62970.0,PBSC_EBIKE,21m 55s,1315266.0,51.51251523,-0.133201961,51.49334336,-0.194757949
17823199,146319210,2025-02-15 00:01:00,1163,"Wardour Street, Soho",2025-02-15 00:23:00,1210.0,"Nevern Place, Earl's Court",63362.0,PBSC_EBIKE,21m 47s,1307848.0,51.51251523,-0.133201961,51.49334336,-0.194757949
17823200,146319211,2025-02-15 00:01:00,1220,"Imperial College, Knightsbridge",2025-02-15 00:27:00,1207.0,"Tavistock Street, Covent Garden",53794.0,CLASSIC,25m 59s,1559056.0,51.49942855,-0.179702476,51.51196803,-0.120718759
17823201,146319212,2025-02-15 00:01:00,2692,"Waterloo Station 2, Waterloo",2025-02-15 01:46:00,200178.0,"Buckingham Gate, Westminster",54259.0,CLASSIC,1h 44m 52s,6292435.0,51.50391973,-0.11342629,51.49886563,-0.137424571
